# 03 · AutoGen：多 Agent 协作框架

当单个 LLM 难以胜任复杂任务时，AutoGen 让多个 AI Agent 相互对话、分工协作——就像一个小型 AI 团队。

**你将学到：**

| 章节 | 内容 |
|------|------|
| 第一章 | 为什么需要多 Agent |
| 第二章 | 核心概念：ConversableAgent、GroupChat |
| 第三章 | 双 Agent 实战：Coder + Critic |
| 第四章 | GroupChat 多角色协作 |
| 第五章 | 工具集成（函数调用） |
| 第六章 | Human-in-the-loop |
| 小结 | 适合场景与局限 |


## 第一章：为什么需要多 Agent？

单个 LLM 的瓶颈：一次对话中难以同时扮演「写代码」「审查代码」「提出测试」等不同角色。

```mermaid
flowchart LR
    subgraph SINGLE["单 Agent 瓶颈"]
        SA[一个 LLM] -->|角色混乱| FAIL["自我审查盲区\n上下文污染"]
    end
    subgraph MULTI["多 Agent 分工"]
        MA1[Coder] -->|提交代码| MA2[Reviewer]
        MA2 -->|提出问题| MA1
        MA2 -->|通过审查| MA3[Tester]
    end
    SINGLE -.->|AutoGen 解决| MULTI
```

**关键洞察：** 每个 Agent 有独立的角色设定和对话历史，可以真正「客观」地评价其他 Agent 的输出。


In [ ]:
# 安装依赖
# pip install pyautogen python-dotenv

import os
from dotenv import load_dotenv
load_dotenv()

import autogen

# Claude 配置
claude_config = {
    'config_list': [{
        'model': 'claude-sonnet-4-6',
        'api_key': os.getenv('ANTHROPIC_API_KEY'),
        'api_type': 'anthropic',
    }],
    'temperature': 0.3,
}
print(f'AutoGen 版本: {autogen.__version__}')


## 第二章：核心概念

```mermaid
flowchart TD
    CA[ConversableAgent] -->|继承| UA[UserProxyAgent]
    CA -->|继承| AA[AssistantAgent]
    UA -->|代表人类| HUMAN[发起对话、审批、终止]
    AA -->|AI 助手| AI[执行任务、生成内容]
    UA & AA -->|加入| GC[GroupChat]
    GC --> GM[GroupChatManager]
    GM -->|协调发言顺序| GC
```

| 类 | 角色 | 关键参数 |
|-----|------|----------|
| `UserProxyAgent` | 代理人类 | `human_input_mode`, `code_execution_config` |
| `AssistantAgent` | AI 助手 | `system_message`, `llm_config` |
| `GroupChat` | 多人对话 | `agents`, `max_round`, `speaker_selection_method` |
| `GroupChatManager` | 协调者 | `groupchat`, `llm_config` |


In [ ]:
# 定义基础 ConversableAgent
assistant = autogen.AssistantAgent(
    name='assistant',
    llm_config=claude_config,
    system_message='You are a helpful AI assistant.',
)

user = autogen.UserProxyAgent(
    name='user',
    human_input_mode='NEVER',   # 自动模式：不需要人类输入
    max_consecutive_auto_reply=3,
    is_termination_msg=lambda x: '任务完成' in x.get('content', ''),
)

# 发起一轮对话
user.initiate_chat(
    assistant,
    message='用一句话解释 Transformer 中 Attention 机制的核心思想。回复完成后说「任务完成」。',
)


## 第三章：双 Agent 实战——Coder + Critic

两个 Agent 相互迭代：Coder 写代码，Critic 审查并提出改进，直到达成共识。


In [ ]:
coder = autogen.AssistantAgent(
    name='Coder',
    llm_config=claude_config,
    system_message=(
        'You are an expert Python programmer. Write clean, efficient code. '
        'When you are done with all revisions, say TERMINATE.'
    ),
)

critic = autogen.AssistantAgent(
    name='Critic',
    llm_config=claude_config,
    system_message=(
        'You are a senior code reviewer. Review code for: correctness, efficiency, edge cases, '
        'and readability. Be constructive. If code is good enough, say APPROVED.'
    ),
)

user_proxy = autogen.UserProxyAgent(
    name='user_proxy',
    human_input_mode='NEVER',
    max_consecutive_auto_reply=6,
    is_termination_msg=lambda x: 'TERMINATE' in x.get('content', '') or 'APPROVED' in x.get('content', ''),
)

# 发起 Coder-Critic 迭代
user_proxy.initiate_chat(
    coder,
    message='Write a Python function that finds all prime numbers up to n using the Sieve of Eratosthenes. Include type hints and docstring.',
)


## 第四章：GroupChat 多角色协作

三个 Agent（研究员、编写者、审稿者）在同一个频道协作完成一篇技术摘要。


In [ ]:
researcher = autogen.AssistantAgent(
    name='Researcher',
    llm_config=claude_config,
    system_message='You are a researcher. Provide key facts and insights on the given topic in 3-5 bullet points.',
)

writer = autogen.AssistantAgent(
    name='Writer',
    llm_config=claude_config,
    system_message='You are a technical writer. Take research points and write a clear 150-word summary.',
)

reviewer = autogen.AssistantAgent(
    name='Reviewer',
    llm_config=claude_config,
    system_message='You are an editor. Review the summary for clarity and accuracy. Say APPROVED if it is good, or suggest specific edits.',
)

group_manager_proxy = autogen.UserProxyAgent(
    name='manager',
    human_input_mode='NEVER',
    max_consecutive_auto_reply=10,
    is_termination_msg=lambda x: 'APPROVED' in x.get('content', ''),
)

groupchat = autogen.GroupChat(
    agents=[group_manager_proxy, researcher, writer, reviewer],
    messages=[],
    max_round=8,
    speaker_selection_method='round_robin',
)
gc_manager = autogen.GroupChatManager(groupchat=groupchat, llm_config=claude_config)

group_manager_proxy.initiate_chat(gc_manager, message='Topic: How does RAG (Retrieval-Augmented Generation) work?')


## 第五章：工具集成（函数调用）

给 Agent 注册 Python 函数，模型可以在对话中自动决定是否调用这些工具：

In [ ]:
import math
from autogen import register_function

tool_assistant = autogen.AssistantAgent(
    name='math_assistant',
    llm_config=claude_config,
    system_message='You are a math assistant. Use the available tools to solve problems. Say TERMINATE when done.',
)

tool_user = autogen.UserProxyAgent(
    name='tool_user',
    human_input_mode='NEVER',
    max_consecutive_auto_reply=5,
    is_termination_msg=lambda x: 'TERMINATE' in x.get('content', ''),
    code_execution_config=False,
)

# 注册工具函数
@register_function(caller=tool_assistant, executor=tool_user, name='calculate', description='Evaluate a math expression')
def calculate(expression: str) -> str:
    try:
        result = eval(expression, {'__builtins__': {}}, {'math': math, 'sqrt': math.sqrt, 'pi': math.pi})
        return str(result)
    except Exception as e:
        return f'Error: {e}'

tool_user.initiate_chat(
    tool_assistant,
    message='What is the area of a circle with radius 7.5? Use pi = 3.14159.',
)


## 第六章：Human-in-the-loop

`human_input_mode='ALWAYS'` 让人类在关键节点介入审批，`'TERMINATE'` 仅在 Agent 请求终止时询问：


In [ ]:
# 演示模式：human_input_mode='NEVER' 模拟自动审批（实际使用时改为 'ALWAYS'）
supervised_user = autogen.UserProxyAgent(
    name='supervised_user',
    human_input_mode='NEVER',     # 改为 'ALWAYS' 即可在每轮后介入
    max_consecutive_auto_reply=2,
    is_termination_msg=lambda x: 'COMPLETE' in x.get('content', ''),
    default_auto_reply='Looks good, continue.',   # NEVER 模式下的自动回复
)

supervised_assistant = autogen.AssistantAgent(
    name='supervised_assistant',
    llm_config=claude_config,
    system_message='Draft a 2-sentence email subject and body. After drafting, say COMPLETE.',
)

supervised_user.initiate_chat(
    supervised_assistant,
    message='Write an email to schedule a team meeting for next Monday at 2pm.',
)
print('\n提示：将 human_input_mode 改为 ALWAYS，可在每轮后手动审批或修改 Agent 的输出。')


## 小结：AutoGen 适合场景与局限

```mermaid
flowchart TD
    NEED[需要多 Agent？] --> Q1{任务类型}
    Q1 -->|代码生成+审查| A1[Coder + Critic 模式]
    Q1 -->|多角色内容生产| A2[GroupChat 模式]
    Q1 -->|需要调用外部工具| A3[Tool + Agent 模式]
    Q1 -->|需要人类监督| A4[Human-in-the-loop 模式]
    Q1 -->|简单单轮任务| A5[直接用 LLM API 即可]
```

**局限：**
- 多轮对话 token 消耗高（每个 Agent 都保留完整对话历史）
- GroupChat 发言顺序的 round_robin 策略不够智能，可能产生无关发言
- 代码执行沙箱安全性需要额外配置（Docker 或受限环境）

**下一章 →** `04_claude_agent_sdk.ipynb`：直接用 Anthropic 原生 SDK 构建 Agent，更轻量、更可控。
